# 05 - BLIP-2：让冻结的 LLM 开眼看世界

学完 BLIP（03 + 04），你已经理解了「看图说话」的基本原理。  
BLIP-2 解决了一个更深层的问题：

> **如果世界上已经有了非常强大的语言模型（GPT、OPT、FlanT5...），  
> 我们能不能直接把它们接上视觉能力，而不用重新训练整个模型？**

BLIP-2 的答案是：**能，而且代价极低。**

---

## BLIP → BLIP-2 的进化

回忆一下 BLIP 的结构：Vision Encoder + Text Decoder，**全部一起训练**。

这有两个问题：
1. **代价高**：每次想用更好的 LLM，就得从头训练整个模型
2. **灾难性遗忘**：联合训练时，LLM 之前学到的知识容易被破坏

BLIP-2 的解法：

```
BLIP（全量训练）                BLIP-2（只训练中间层）

[图像编码器]  ←训练→            [图像编码器]  ←冻结→
     ↓                               ↓
[文本解码器]  ←训练→            [ Q-Former ]  ←只训练这里！约 188M 参数
                                     ↓
                               [  大型 LLM  ]  ←冻结→
                               (OPT / FlanT5)
```

**Q-Former（Querying Transformer）是 BLIP-2 的核心创新。**  
它是一个轻量级的「翻译官」，把视觉信息转化成 LLM 能理解的语言表示。

### 直觉理解

想象这样一个场景：
- 你有一个只会看图的专家（ViT，13亿参数，冻结）  
- 你有一个只会说话的专家（OPT，27亿参数，冻结）  
- 你雇了一个翻译官（Q-Former，1.88亿参数，需要训练）

翻译官的工作：从图片里提取最重要的信息（用 32 个「问题 token」去图里查询），然后翻译成 LLM 能读懂的格式。

只需要训练翻译官，两位专家的知识完整保留。而且 LLM 升级了（比如从 OPT-2.7B 换到 GPT-4），只需要重新训练翻译官！

---

## 本 notebook 的内容

```
Step 0  准备环境
Step 1  加载 BLIP-2 模型
Step 2  准备测试图片
Step 3  图像描述生成
Step 4  视觉问答（VQA）
Step 5  Prompting 的技巧
Step 6  BLIP-1 vs BLIP-2 直接对比
```

## Step 0：准备环境

> **内存提醒**：BLIP-2 (OPT-2.7B) 用 float16 加载约需 **6 GB 显存**。  
> 没有 GPU 也能运行，但会很慢（CPU 推理单张图约需 1-2 分钟）。

In [ ]:
import os
import requests
from pathlib import Path

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from transformers import (
    Blip2Processor,
    Blip2ForConditionalGeneration,
    BlipProcessor,                      # 后面 BLIP-1 vs BLIP-2 对比用
    BlipForConditionalGeneration,
)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"设备:    {device}")

# float16 在 GPU 上大幅节省显存；CPU 上只能用 float32
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"精度:    {DTYPE}")

## Step 1：加载 BLIP-2 模型

我们使用 `Salesforce/blip2-opt-2.7b`：
- **ViT-g/14**（图像编码器，13亿参数，冻结）  
- **Q-Former**（翻译官，1.88亿参数）  
- **OPT-2.7B**（语言模型，27亿参数，冻结）

总参数约 **3.9B**，是 BLIP-1 base（990M）的 4 倍，但生成质量远远更强。

> 如果你的 GPU 显存充裕（>= 16 GB），可以改用 `blip2-opt-6.7b` 或 `blip2-flan-t5-xl`。

In [ ]:
BLIP2_NAME  = "Salesforce/blip2-opt-2.7b"
BLIP2_CACHE = "./cache/models/blip2"
os.makedirs(BLIP2_CACHE, exist_ok=True)

# 检查本地缓存
slug = BLIP2_NAME.replace("/", "--")
cached = Path(BLIP2_CACHE).exists() and any(Path(BLIP2_CACHE).glob(f"models--{slug}*"))

if cached:
    print("[缓存命中] 从本地加载 BLIP-2")
else:
    print("[开始下载] BLIP-2 约 16 GB (float32) / 8 GB (float16 shards)")
    print("  如果下载卡住，检查代理后重新运行此 cell（支持断点续传）")

processor2 = Blip2Processor.from_pretrained(BLIP2_NAME, cache_dir=BLIP2_CACHE)

model2 = Blip2ForConditionalGeneration.from_pretrained(
    BLIP2_NAME,
    torch_dtype=DTYPE,      # float16 约 6GB 显存，float32 约 16GB
    cache_dir=BLIP2_CACHE,
).to(device)
model2.eval()

n_params = sum(p.numel() for p in model2.parameters()) / 1e9
print(f"\n模型加载完成！总参数量: {n_params:.2f}B")

# 展示三个组件的参数量分布
components = {
    "vision_model (冻结)": model2.vision_model,
    "qformer (可训练)": model2.qformer,
    "language_model (冻结)": model2.language_model,
}
print("\n各组件参数量：")
for name, module in components.items():
    n = sum(p.numel() for p in module.parameters()) / 1e6
    print(f"  {name:30s}: {n:.0f}M")

## Step 2：准备测试图片

复用 `03_blip_basics` 里的两张图，再加几张场景不同的，覆盖更多情况。  
下载时先检查本地缓存，有就直接用。

In [ ]:
IMAGE_DIR = "./cache/images"
os.makedirs(IMAGE_DIR, exist_ok=True)

def load_image(url: str, filename: str) -> Image.Image:
    """下载并缓存图片，已存在则直接读取"""
    path = os.path.join(IMAGE_DIR, filename)
    if os.path.exists(path):
        print(f"[缓存] {filename}")
    else:
        print(f"[下载] {filename} ...")
        try:
            r = requests.get(url, timeout=30, stream=True)
            r.raise_for_status()
            with open(path, "wb") as f:
                for chunk in r.iter_content(8192):
                    f.write(chunk)
        except Exception as e:
            raise RuntimeError(f"下载失败: {e}\n请检查代理或手动下载图片到 {path}")
    return Image.open(path).convert("RGB")


images = {
    "海滩女孩与狗": load_image(
        "https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg",
        "demo.jpg",
    ),
    "胖猫": load_image(
        "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg",
        "cat.jpg",
    ),
    "城市夜景": load_image(
        "https://images.unsplash.com/photo-1477959858617-67f85cf4f1df?w=800",
        "city_night.jpg",
    ),
    "咖啡拉花": load_image(
        "https://images.unsplash.com/photo-1509042239860-f550ce710b93?w=800",
        "coffee.jpg",
    ),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img)
    ax.set_title(name, fontsize=10)
    ax.axis("off")
plt.suptitle("测试图片", fontsize=12)
plt.tight_layout()
plt.show()

## Step 3：图像描述生成

最简单的用法：只给图片，让模型自由描述。

和 BLIP-1 的调用方式很像，但有一个关键区别：  
输入张量需要转成和模型一致的精度（`.to(device, DTYPE)`）。

In [ ]:
def caption(model, proc, image, device, dtype, max_new_tokens=50, num_beams=5):
    """无条件图像描述生成"""
    inputs = proc(images=image, return_tensors="pt").to(device, dtype)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=num_beams)
    return proc.batch_decode(ids, skip_special_tokens=True)[0].strip()


print("=== BLIP-2 图像描述 ===")
for name, img in images.items():
    cap = caption(model2, processor2, img, device, DTYPE)
    print(f"\n【{name}】\n  {cap}")

## Step 4：视觉问答（VQA）

BLIP-2 的 VQA 和 BLIP-1 不同：
- BLIP-1 有专门的 `BlipForQuestionAnswering` 类，答案通常很短
- BLIP-2 直接用语言模型生成，**用 prompt 格式引导**，回答更自然、更详细

标准 VQA prompt 格式：
```
Question: {你的问题} Answer:
```

模型会续写 `Answer:` 后面的内容。

In [ ]:
def vqa(model, proc, image, question, device, dtype, max_new_tokens=30):
    """视觉问答。返回 (完整输出, 仅答案部分)"""
    prompt = f"Question: {question} Answer:"
    inputs = proc(images=image, text=prompt, return_tensors="pt").to(device, dtype)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    full  = proc.batch_decode(ids, skip_special_tokens=True)[0].strip()
    # 截取 "Answer:" 之后的部分
    answer = full.split("Answer:")[-1].strip()
    return answer


img = images["海滩女孩与狗"]
questions = [
    "What is the woman doing?",
    "What animal is with the woman?",
    "What time of day does it appear to be?",
    "Describe the mood of this scene.",
    "If you had to write a news headline for this photo, what would it be?",
]

print("=== 对「海滩女孩与狗」提问 ===\n")
for q in questions:
    ans = vqa(model2, processor2, img, q, device, DTYPE)
    print(f"Q: {q}")
    print(f"A: {ans}\n")

In [ ]:
# 再试试其他图片
img_coffee = images["咖啡拉花"]
questions_coffee = [
    "What drink is this?",
    "What art is on top of the drink?",
    "What kind of shop would serve this?",
    "Can you estimate the price of this drink?",
]

print("=== 对「咖啡拉花」提问 ===\n")
for q in questions_coffee:
    ans = vqa(model2, processor2, img_coffee, q, device, DTYPE)
    print(f"Q: {q}")
    print(f"A: {ans}\n")

## Step 5：Prompting 的技巧

BLIP-2 底层是一个真正的 LLM，这意味着**你怎么问，它就怎么回答**。  
不同的 prompt 会引导模型产生风格截然不同的输出。

### 5.1 不同 prompt 风格对比

In [ ]:
def prompted_caption(model, proc, image, prompt, device, dtype, max_new_tokens=60):
    inputs = proc(images=image, text=prompt, return_tensors="pt").to(device, dtype)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=5)
    full = proc.batch_decode(ids, skip_special_tokens=True)[0].strip()
    # 去掉 prompt 前缀
    return full[len(prompt):].strip() if full.startswith(prompt) else full


img = images["城市夜景"]
prompts = [
    ("无 prompt（自由生成）",        ""),
    ("简洁描述",                     "A photo of"),
    ("详细描述",                     "Describe this image in detail:"),
    ("诗意风格",                     "Write a poetic description of this scene:"),
    ("社交媒体 caption",             "Instagram caption for this photo:"),
    ("新闻标题",                     "A newspaper headline for this image:"),
]

print("=== 同一张图，不同 prompt 的效果 ===\n")
for label, p in prompts:
    if p == "":  # 无 prompt 走 caption 函数
        result = caption(model2, processor2, img, device, DTYPE)
    else:
        result = prompted_caption(model2, processor2, img, p, device, DTYPE)
    print(f"[{label}]")
    print(f"  {result}\n")

### 5.2 多轮对话（延续上下文）

因为有 LLM 在背后，BLIP-2 支持把前面的问答作为上下文继续提问。  
把已有的问答拼接成字符串，就像在和 LLM 对话一样。

In [ ]:
img = images["海滩女孩与狗"]

# 对话历史：一问一答交替拼接
conversation = ""
turns = [
    "What can you see in this image?",
    "What is the relationship between the woman and the dog?",
    "What do you think they are feeling?",
]

print("=== 多轮对话 ===\n")
for question in turns:
    # 把历史上下文 + 新问题拼在一起
    prompt = conversation + f"Question: {question} Answer:"
    inputs = processor2(images=img, text=prompt, return_tensors="pt").to(device, DTYPE)
    with torch.no_grad():
        ids = model2.generate(**inputs, max_new_tokens=40)
    full   = processor2.batch_decode(ids, skip_special_tokens=True)[0].strip()
    answer = full.split("Answer:")[-1].strip()

    print(f"Q: {question}")
    print(f"A: {answer}\n")

    # 把这轮问答加入历史，下一轮可以引用
    conversation += f"Question: {question} Answer: {answer}\n"

## Step 6：BLIP-1 vs BLIP-2 直接对比

用同一张图片，分别跑 BLIP-1 和 BLIP-2，并排比较输出质量。  
你会直观感受到：BLIP-2 的语言更自然、描述更详细、回答更开放。

> 这里会同时加载两个模型，总共约需 **7~8 GB 显存**。  
> 如果 OOM，可以先 `del model2` 释放后再加载 BLIP-1。

In [ ]:
BLIP1_NAME  = "Salesforce/blip-image-captioning-base"
BLIP1_CACHE = "./cache/models/blip-caption"

blip1_slug   = BLIP1_NAME.replace("/", "--")
blip1_cached = Path(BLIP1_CACHE).exists() and any(Path(BLIP1_CACHE).glob(f"models--{blip1_slug}*"))
print("[缓存命中]" if blip1_cached else "[下载中]", "BLIP-1")

processor1 = BlipProcessor.from_pretrained(BLIP1_NAME, cache_dir=BLIP1_CACHE)
model1     = BlipForConditionalGeneration.from_pretrained(
    BLIP1_NAME, cache_dir=BLIP1_CACHE,
).to(device)
model1.eval()
print("BLIP-1 加载完成")

In [ ]:
def blip1_caption(image, question=None):
    """BLIP-1 生成描述或回答问题"""
    inp = processor1(images=image, text=question, return_tensors="pt").to(device)
    with torch.no_grad():
        ids = model1.generate(**inp, max_length=50, num_beams=5)
    return processor1.decode(ids[0], skip_special_tokens=True)


# 对比四张图的描述
fig, axes = plt.subplots(len(images), 3, figsize=(16, 4.5 * len(images)))

col_labels = ["图片", "BLIP-1 (990M)", "BLIP-2 (3.9B, OPT)"]
for ax, label in zip(axes[0], col_labels):
    ax.set_title(label, fontsize=11, fontweight="bold")

for row, (name, img) in enumerate(images.items()):
    cap1 = blip1_caption(img)
    cap2 = caption(model2, processor2, img, device, DTYPE)

    for col, (ax, text) in enumerate(zip(axes[row], [name, cap1, cap2])):
        ax.imshow(img)
        ax.axis("off")
        if col == 0:
            ax.set_xlabel(f"图片：{name}", fontsize=9)
        else:
            # 折行，每行最多 50 字符
            lines = "\n".join(text[i:i+50] for i in range(0, len(text), 50))
            ax.set_xlabel(lines, fontsize=7.5)

plt.suptitle("BLIP-1 vs BLIP-2 描述效果对比", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# VQA 对比：BLIP-1（专用 VQA 模型）vs BLIP-2（通用生成）
from transformers import BlipForQuestionAnswering

BLIP1_VQA_NAME  = "Salesforce/blip-vqa-base"
BLIP1_VQA_CACHE = "./cache/models/blip-vqa"
os.makedirs(BLIP1_VQA_CACHE, exist_ok=True)

vqa1_slug   = BLIP1_VQA_NAME.replace("/", "--")
vqa1_cached = Path(BLIP1_VQA_CACHE).exists() and any(Path(BLIP1_VQA_CACHE).glob(f"models--{vqa1_slug}*"))
print("[缓存命中]" if vqa1_cached else "[下载中]", "BLIP-1 VQA")

proc_vqa1  = BlipProcessor.from_pretrained(BLIP1_VQA_NAME, cache_dir=BLIP1_VQA_CACHE)
model_vqa1 = BlipForQuestionAnswering.from_pretrained(
    BLIP1_VQA_NAME, cache_dir=BLIP1_VQA_CACHE,
).to(device)
model_vqa1.eval()
print("BLIP-1 VQA 加载完成")

In [ ]:
img  = images["海滩女孩与狗"]
qs   = [
    "What is the woman wearing?",
    "What is the dog doing?",
    "How does the woman feel?",
    "Write a short story about this image.",  # 这个问题只有 LLM 能答
]

print(f"{'问题':<40} {'BLIP-1 VQA':<30} {'BLIP-2 (OPT)'}")
print("-" * 110)

for q in qs:
    # BLIP-1 VQA
    inp1 = proc_vqa1(images=img, text=q, return_tensors="pt").to(device)
    with torch.no_grad():
        ids1 = model_vqa1.generate(**inp1, max_length=20)
    ans1 = proc_vqa1.decode(ids1[0], skip_special_tokens=True)

    # BLIP-2
    ans2 = vqa(model2, processor2, img, q, device, DTYPE, max_new_tokens=50)

    print(f"{q[:38]:<40} {ans1[:28]:<30} {ans2[:50]}")

## 可视化：Q-Former 的作用

Q-Former 用 **32 个可学习的 query token** 去「查询」图片，提取最关键的信息。  
我们可以打印出这 32 个 token 的维度，直观感受它在架构中的位置。

In [ ]:
print("Q-Former 结构一览：")
print(f"  可学习 query tokens 数量: {model2.query_tokens.shape[1]}")
print(f"  query 维度:              {model2.query_tokens.shape[2]}")
print()

# 手动跑一次 forward，看 Q-Former 输出的形状
img_sample = list(images.values())[0]
inputs_vis = processor2(images=img_sample, return_tensors="pt").to(device, DTYPE)

with torch.no_grad():
    vision_outputs = model2.vision_model(
        pixel_values=inputs_vis["pixel_values"],
        output_attentions=False,
        output_hidden_states=False,
    )
    image_embeds = vision_outputs[0]  # [1, num_patches, hidden_dim]

print("数据流维度：")
print(f"  输入图片:          {inputs_vis['pixel_values'].shape}  (B, C, H, W)")
print(f"  ViT 输出 patches:  {image_embeds.shape}  (B, num_patches, vision_dim)")
print(f"  Q-Former 输入:     32 个 query token，每个维度 {model2.query_tokens.shape[2]}")
print(f"  Q-Former 输出:     32 × {model2.config.qformer_config.hidden_size}  →  送入 LLM")
print()
print("对比：")
n_patches = image_embeds.shape[1]
n_queries = model2.query_tokens.shape[1]
print(f"  ViT 产生 {n_patches} 个 patch token")
print(f"  Q-Former 压缩为 {n_queries} 个 query token  （压缩比 {n_patches/n_queries:.1f}x）")
print(f"  LLM 只需要处理 {n_queries} 个额外 token，负担极小")

## 总结

### BLIP-1 vs BLIP-2 对比

| | BLIP-1 | BLIP-2 |
|---|---|---|
| 架构 | Vision Encoder + Text Decoder（全部训练） | 冻结 ViT + Q-Former + 冻结 LLM |
| 核心创新 | CapFilt 自举数据清洗 | Q-Former 轻量级视觉-语言桥接 |
| 可训练参数 | ~990M（全部） | ~188M（只有 Q-Former） |
| 语言能力 | 自己的 text decoder | 继承任何已有 LLM 的能力 |
| VQA 风格 | 短答案（分类式） | 开放式生成，更自然 |
| 多轮对话 | 不支持 | 支持（把历史上下文拼接即可） |
| 可扩展性 | LLM 升级需重训 | 换更强的 LLM 只需重训 Q-Former |

### Q-Former 的核心思想

```
图片（256 个 patch）
    │
    ▼ 冻结的 ViT
    │  256 × 1408
    │
    ▼ Q-Former（只训练这里）
    │  32 × 768     ← 32 个 query token 去图里查询最重要的信息
    │
    ▼ 线性投影
    │  32 × LLM_dim
    │
    ▼ 冻结的 LLM（OPT / FlanT5）
    │
    ▼ 生成文本
```

### 下一步

- 换用 `blip2-flan-t5-xl`：基于指令调优的 FlanT5，更擅长按格式回答
- 了解 **InstructBLIP**：在 BLIP-2 基础上加入指令微调，对话能力更强
- 了解 **LLaVA**：用更简单的线性投影替代 Q-Former，效果却不差（下一个 notebook）